# T1.1 — Anti-spoof model smoke test

Verify `MelodyMachine/Deepfake-audio-detection-V2` (a) loads under
200 MB, (b) returns class probabilities on 16 kHz mono float32 input,
(c) separates real speech from SpeechT5-generated speech on a histogram.

Corpus: 10 real clips from LibriSpeech test-clean + 10 SpeechT5 clips
generated in this notebook.

Acceptance (see docs/TASKS.md T1.1): model loads, logits present, real
vs. synthetic histograms show visible separation (not necessarily
publication-grade).

In [ ]:
# --- VishGuard setup cell (copy from notebooks/00_colabSetup.ipynb) ---
import os, sys, subprocess
from pathlib import Path
REPO_URL = os.environ.get('VISHGUARD_REPO_URL', '')
DRIVE_SRC = '/content/drive/MyDrive/vishguard'
REPO_DIR = Path('/content/vishguard')
ON_COLAB = 'google.colab' in sys.modules or Path('/content').exists()
def sh(cmd, check=True):
    print('$', cmd); subprocess.run(cmd, shell=True, check=check)
if ON_COLAB:
    try:
        from google.colab import drive; drive.mount('/content/drive', force_remount=False)
    except Exception as e: print('Drive mount skipped:', e)
    if REPO_URL:
        if REPO_DIR.exists(): sh(f'cd {REPO_DIR} && git pull --ff-only')
        else: sh(f'git clone {REPO_URL} {REPO_DIR}')
    elif Path(DRIVE_SRC).exists():
        REPO_DIR.mkdir(parents=True, exist_ok=True)
        sh(f'rsync -a --delete --exclude .venv --exclude __pycache__ {DRIVE_SRC}/ {REPO_DIR}/')
    else:
        raise RuntimeError('Set VISHGUARD_REPO_URL or copy the repo to MyDrive/vishguard/')
    sh(f'pip install -q -e {REPO_DIR}')
    sh(f'pip install -q -r {REPO_DIR}/requirements.txt')
    src = str(REPO_DIR / 'src')
    if src not in sys.path: sys.path.insert(0, src)
import vishguard; print('vishguard', vishguard.__version__)

In [ ]:
import numpy as np
import torch
from transformers import pipeline

SPOOF_MODEL_ID = 'MelodyMachine/Deepfake-audio-detection-V2'
device = 0 if torch.cuda.is_available() else -1
spoof = pipeline('audio-classification', model=SPOOF_MODEL_ID, device=device)
print('labels:', spoof.model.config.id2label)
# Sanity: model should be small. Print param count in MB.
params = sum(p.numel() for p in spoof.model.parameters())
print(f'params={params/1e6:.1f}M  fp32-size={(params*4)/1e6:.0f} MB')

In [ ]:
# Real speech: 10 LibriSpeech test-clean clips via HF datasets (streaming).
from datasets import load_dataset
import librosa

real_clips = []
ds = load_dataset('openslr/librispeech_asr', 'clean', split='test', streaming=True,
                  trust_remote_code=True)
for i, ex in enumerate(ds):
    if i >= 10: break
    audio = ex['audio']
    samples = audio['array'].astype(np.float32)
    sr = audio['sampling_rate']
    if sr != 16_000:
        samples = librosa.resample(samples, orig_sr=sr, target_sr=16_000)
    real_clips.append(samples)
print(f'real_clips: {len(real_clips)} (sample shapes: {[c.shape for c in real_clips[:3]]})')

In [ ]:
# Synthetic speech: 10 SpeechT5 clips (also reused for the T3.1 eval corpus).
from transformers import SpeechT5Processor, SpeechT5ForTextToSpeech, SpeechT5HifiGan
from datasets import load_dataset as _ld

tts_proc = SpeechT5Processor.from_pretrained('microsoft/speecht5_tts')
tts_model = SpeechT5ForTextToSpeech.from_pretrained('microsoft/speecht5_tts')
vocoder = SpeechT5HifiGan.from_pretrained('microsoft/speecht5_hifigan')
embed_ds = _ld('Matthijs/cmu-arctic-xvectors', split='validation')
speaker = torch.tensor(embed_ds[7306]['xvector']).unsqueeze(0)

scripts = [
    'Hello, this is an automated message from your bank.',
    'We detected unusual activity on your account. Please confirm your identity.',
    'Your package delivery has been rescheduled. Press one to continue.',
    'This is the IRS. You have an outstanding tax balance.',
    'Your computer has been compromised. Do not turn it off.',
    'Congratulations, you have won a five hundred dollar gift card.',
    'This is your car warranty specialist calling.',
    'Please verify the last four digits of your social security number.',
    'Your subscription will renew automatically tomorrow.',
    'Press one to speak with a representative about your loan.',
]
synth_clips = []
for text in scripts:
    inputs = tts_proc(text=text, return_tensors='pt')
    with torch.no_grad():
        wav = tts_model.generate_speech(inputs['input_ids'], speaker, vocoder=vocoder)
    synth_clips.append(wav.numpy().astype(np.float32))
print(f'synth_clips: {len(synth_clips)} (shapes: {[c.shape for c in synth_clips[:3]]})')

In [ ]:
# Run detector on both sets. Capture pSynthetic per clip.
def p_synthetic(clip):
    preds = spoof({'array': clip, 'sampling_rate': 16_000}, top_k=None)
    # Find label corresponding to fake/synthetic. Model's label set is logged above.
    for p in preds:
        if 'fake' in p['label'].lower() or 'spoof' in p['label'].lower() or 'synthetic' in p['label'].lower():
            return p['score']
    # Fallback: assume label index 1 is synthetic.
    return preds[1]['score']

real_scores = [p_synthetic(c) for c in real_clips]
synth_scores = [p_synthetic(c) for c in synth_clips]
print('real   p(synth):', [round(s, 3) for s in real_scores])
print('synth  p(synth):', [round(s, 3) for s in synth_scores])

In [ ]:
import matplotlib.pyplot as plt
plt.figure(figsize=(7, 4))
plt.hist(real_scores, bins=10, alpha=0.6, label='real speech (LibriSpeech)')
plt.hist(synth_scores, bins=10, alpha=0.6, label='SpeechT5 synthetic')
plt.axvline(0.5, color='k', ls='--', lw=1)
plt.xlabel('p(synthetic)'); plt.ylabel('count')
plt.title('T1.1 — MelodyMachine/Deepfake-audio-detection-V2')
plt.legend(); plt.tight_layout(); plt.show()
import numpy as np
print(f'mean real={np.mean(real_scores):.3f}  mean synth={np.mean(synth_scores):.3f}')
print(f'separation at 0.5 threshold: real<0.5={sum(s<0.5 for s in real_scores)}/10, synth>=0.5={sum(s>=0.5 for s in synth_scores)}/10')

## Acceptance checklist

- [ ] Model loads and prints id2label
- [ ] Params under ~200 MB fp32
- [ ] `p_synthetic` defined in [0, 1] for all 20 clips
- [ ] Histograms visibly separated; record means and threshold-0.5 counts

If this fails, the fallback path is Alt A (`motheecreator/Deepfake-audio-detection`) — see docs/ARCHITECTURE.md §2.2.